# CPG Schema, Models, and Graph Diff

This notebook demonstrates:
- The full Joern CPG schema (20 node types, 14+ edge types, 9 languages)
- Parsing Joern exports with `CPGNode.from_joern()` and `CPGEdge.from_joern()`
- Graph analysis: type distribution, call graph, structure
- Change detection with `GraphDiff`

**Prerequisites:**
- `pip install graphrag-toolkit-codeproperty-graph`
- No Neptune or AWS credentials required


## 1. The CPG Schema

Explore the complete Joern type system.


In [ ]:
from graphrag_toolkit.codeproperty_graph import (
    NodeType, EdgeType, DELTA_RELEVANT_TYPES, SUPPORTED_LANGUAGES,
    joern_export_command
)

print(f'Node types ({len(NodeType)}):')
for nt in NodeType:
    marker = ' ★' if nt in DELTA_RELEVANT_TYPES else ''
    print(f'  {nt.value:20s}{marker}')

print(f'\nEdge types ({len(EdgeType)}):')
for et in EdgeType:
    print(f'  {et.value}')

print(f'\nSupported languages ({len(SUPPORTED_LANGUAGES)}):')
for lang, desc in SUPPORTED_LANGUAGES.items():
    print(f'  {lang:12s} {desc}')

print(f'\n★ = participates in delta comparison')


## 2. Joern Export Command

Generate the CLI command to produce a CPG from source code.


In [ ]:
# Generate export commands for different languages
for lang in ['java', 'python', 'javascript']:
    cmd = joern_export_command('/src/my-service', output_dir='cpg-out', language=lang)
    print(f'{lang:12s} {cmd}')

# Auto-detect language
print(f'{"auto":12s} {joern_export_command("/src/my-service")}')


## 3. Parse Joern Export with `from_joern()`

Load raw Joern JSON and parse into typed CPGNode/CPGEdge objects.
The factory methods handle both UPPER_CASE and camelCase property formats.


In [ ]:
import json
from graphrag_toolkit.codeproperty_graph import CPGNode, CPGEdge

with open('data/sample_nodes.json') as f:
    raw_nodes = json.load(f)
with open('data/sample_edges.json') as f:
    raw_edges = json.load(f)

# Parse with from_joern()
nodes = [CPGNode.from_joern(raw) for raw in raw_nodes]
edges = [CPGEdge.from_joern(raw) for raw in raw_edges]

print(f'Parsed {len(nodes)} nodes, {len(edges)} edges')


## 4. Graph Analysis: Type Distribution

See what node and edge types are in the graph.


In [ ]:
from collections import Counter

# Node type distribution
node_types = Counter(n.node_type for n in nodes)
print('Node types:')
for ntype, count in node_types.most_common():
    print(f'  {ntype:20s} {count:3d}')

# Edge type distribution
edge_types = Counter(e.edge_type for e in edges)
print(f'\nEdge types:')
for etype, count in edge_types.most_common():
    print(f'  {etype:20s} {count:3d}')


## 5. Explore Methods (Delta-Relevant Nodes)

Only METHOD nodes participate in change detection.
Their `full_name → hash` mapping is the fingerprint for delta comparison.


In [ ]:
methods = [n for n in nodes if n.node_type == NodeType.METHOD]

print(f'Methods ({len(methods)}):')
for m in methods:
    print(f'  {m.full_name}')
    print(f'    signature:  {m.signature}')
    print(f'    hash:       {m.hash}')
    print(f'    file:       {m.filename}:{m.line_number}-{m.line_number_end}')
    print(f'    external:   {m.is_external}')
    print()


## 6. Explore Structure (Classes, Files, Namespaces)


In [ ]:
from graphrag_toolkit.codeproperty_graph.schema import STRUCTURE_TYPES

structure_nodes = [n for n in nodes if n.node_type in STRUCTURE_TYPES]
print(f'Structure nodes ({len(structure_nodes)}):')
for n in structure_nodes:
    print(f'  {n.node_type:16s} {n.full_name or n.name}')


## 7. Call Graph

Extract the CALL edges to see which methods call which.


In [ ]:
call_edges = [e for e in edges if e.edge_type == EdgeType.CALL]
node_map = {n.id: n for n in nodes}

print(f'Call graph ({len(call_edges)} edges):')
for e in call_edges:
    src = node_map.get(e.source_id)
    tgt = node_map.get(e.target_id)
    src_name = src.full_name if src else e.source_id
    tgt_name = tgt.full_name if tgt else e.target_id
    print(f'  {src_name} → {tgt_name}')


## 8. Delta Comparison (GraphDiff)

Compare method signatures between two CPG snapshots.
This is how `DeltaIngestor` decides whether to skip or ingest.


In [ ]:
from graphrag_toolkit.codeproperty_graph import GraphDiff

# Current state
current_sigs = {m.full_name: m.hash for m in methods}
print(f'Current signatures:')
for name, h in current_sigs.items():
    print(f'  {name}: {h}')

# Simulate next commit: parse() refactored, validate() deleted, check() added
next_sigs = {
    'com.example.Main.main': 'a1b2c3d4e5',       # unchanged
    'com.example.Parser.parse': 'NEWH4SH999',     # MODIFIED
    'com.example.Checker.check': 'p6q7r8s9t0',    # ADDED
}

diff = GraphDiff.compare(current_sigs, next_sigs)
print(f'\nDelta: {diff.summary}')
print(f'  Added:     {list(diff.added.keys())}')
print(f'  Removed:   {list(diff.removed.keys())}')
print(f'  Modified:  {list(diff.modified.keys())}')
print(f'  Unchanged: {diff.unchanged}')
print(f'\n→ DeltaIngestor would: {"INGEST" if diff.has_changes else "SKIP"}')


## 9. No-Change Case (SKIP)

When code hasn't changed, `GraphDiff` returns no delta — zero Neptune writes.


In [ ]:
no_diff = GraphDiff.compare(current_sigs, current_sigs)
print(f'Delta: {no_diff.summary}')
print(f'Has changes: {no_diff.has_changes}')
print(f'\n→ DeltaIngestor would: SKIP')
print(f'  Zero Neptune writes. Zero cost. <1 second.')


## Summary

| Feature | What It Does |
|---------|-------------|
| `NodeType` / `EdgeType` enums | Complete Joern schema (20 + 14 types) |
| `CPGNode.from_joern(raw)` | Parse Joern JSON export → typed objects |
| `DELTA_RELEVANT_TYPES` | Which types matter for change detection |
| `STRUCTURE_TYPES` | Types for architectural analysis |
| `GraphDiff.compare()` | Method signature delta → skip or ingest |
| `joern_export_command()` | Generate the CLI command for any language |
